# Penyisihan IFEST 2026 DAC — v6: Synthetic Negative Augmentation

## Root cause analysis (this is why v2-v5 plateaued at ~0.75)

Direct inspection of the 874 `content_hash` groups that carry **both** labels shows how
the negative class was actually built:

```
tokens differing between the positive title and the negative title (same article body)
  0 tokens : 186   (whitespace variants)
  1 token  : 529   <-- 48% of all pairs
  2 tokens :  76
  ...tail (5-11 tokens): ~280  (a wholly different headline)
```

Of the 468 pairs that differ by exactly one token swapped for one token:
- **95% of the swapped-out tokens are proper nouns** — `DKI`->`Jabar`, `Bogor`->`Bandung`,
  `Karawang`->`Bekasi`, `Jokowi`->`Prabowo`, `Istana`->`Bio Farma`, `China`->`Malaysia`
- **96% of the swapped-in tokens appear elsewhere in the corpus** — they are drawn from
  the dataset's own entity pool
- **0 exact `(title, content)` pairs ever carry conflicting labels** — there is no label
  noise, so the exact-match override is completely sound

**The negative class is generated by swapping one proper noun in a true headline.**

That single fact explains every symptom we hit:

| Symptom | Explanation |
|---|---|
| F1 class1 = 0.95 but F1 class0 = 0.54 | "is this headline about this article" is easy; "was exactly one entity tampered with" is a different, much finer task |
| Lexical/semantic similarity features useless | a tampered headline is ~95% identical to a true one |
| Threshold sweep flat across 0.2-0.7 | the score distributions genuinely overlap — a representation problem, not calibration |
| **v5's cross-article mining LOST 0.012** | it taught "is this a different article" (already easy) instead of "was one entity swapped" — actively off-task |

A corroborating measurement: **a title entity that is absent from the body occurs in 38%
of negatives vs 9.8% of positives (3.9x lift)**. Used alone as a hand-written rule it
scores **Macro F1 = 0.6263** — better than the entire TF-IDF baseline (0.52).

## What v6 does about it

1. **Synthetic negative augmentation (the core fix).** We only have 1,437 real negatives
   against 12,960 positives — class 0 is *data-starved*. We generate new negatives the
   same way the organizers did: take a positive, find a title token that is a genuine
   proper noun *and* is grounded in the body, and replace it with a type-compatible
   entity that does **not** appear in the body (so the new label is reliably 0).
   - **Purity filter**: a real proper noun rarely appears lowercase in the corpus.
     Measured: `DKI`=1.00, `Jabar`=1.00, `Jokowi`=0.99 (kept) vs `Tidak`=0.05,
     `Jenazah`=0.08, `Corona`=0.44 (rejected).
   - **Distributional type matching**: entities sharing a preceding word in headlines are
     swap-compatible. Learned automatically — after `Gubernur`: provinces; after `Pemkot`:
     cities; after `Kapolda`: police regions.
   - Measured 75% generation success, 78% of those type-matched.
2. **The missing conflict signal.** Previous versions listed body-entities missing from
   the title (weak). v6 adds the *discriminative* direction measured above — **title
   entities missing from the body** — into the model's input text.
3. **Removed: cross-article mining** (v5, cost -0.012 on the leaderboard) and the
   contrastive phase. Augmentation attacks the same problem far more directly and at ~10x
   the scale, and a simpler pipeline has fewer ways to fail.
4. **Kept** (all validated): hybrid overlapping retrieval, exact-match override,
   threshold tuning, `GroupShuffleSplit` on `content_hash`.

**Validation integrity:** synthetic rows are added to the *training* split only. The
validation split keeps the original ~90/10 distribution, so threshold tuning and model
selection still reflect the real test distribution.

## 1. Environment

In [ ]:
import os, sys, subprocess
def _pip(pkgs):
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True, timeout=180)
    except Exception as e:
        print(f"[warn] pip install {pkgs} failed/skipped: {e}")

_pip(['-U', 'transformers', 'accelerate'])

In [ ]:
import re, time, json, hashlib, unicodedata, warnings, random, string as _string, shutil
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers
print("transformers:", transformers.__version__)

TIMINGS = {}
class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *a):
        dt = time.time() - self.t0
        TIMINGS[self.name] = TIMINGS.get(self.name, 0) + dt
        print(f"[TIMER] {self.name}: {dt:.1f}s")

RUN_T0 = time.time()
def gpu_mem_mb():
    return torch.cuda.max_memory_allocated()/1e6 if torch.cuda.is_available() else 0.0
def rm_checkpoint(p):
    shutil.rmtree(p, ignore_errors=True); print(f"[cleanup] removed {p}")

## 2. Configuration

In [ ]:
CONFIG = {
    'seed': SEED,
    # retrieval
    'retriever_model': 'intfloat/multilingual-e5-small',
    'chunk_words': 50, 'chunk_stride': 25,
    'top_k_semantic': 3, 'top_k_lexical': 2, 'include_first_chunk': True,
    'retriever_max_len': 96, 'retriever_batch_size': 256, 'lexical_max_features': 60000,
    # classifier
    'classifier_model': 'indobenchmark/indobert-base-p1',
    'max_len': 384, 'batch_size': 16, 'lr': 2e-5, 'epochs': 3,
    'early_stopping_patience': 1, 'val_size': 0.15,
    # synthetic negative augmentation
    'entity_min_count': 8,        # min corpus occurrences to be considered an entity
    'entity_min_purity': 0.95,    # capitalized/(capitalized+lowercase) -- rejects common words
    'entity_max_title_df': 0.02,  # drop ubiquitous topic words (Corona/Covid)
    'target_neg_share': 0.40,     # desired class-0 share of the TRAINING split after augmentation
    # thresholding
    'threshold_grid': [round(x,2) for x in np.arange(0.20, 0.81, 0.05)],
}
print(json.dumps(CONFIG, indent=2, default=str))

## 3. Load Data

In [ ]:
with Timer("data_loading"):
    DATA_FILES = {}
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f in ('train.csv','test.csv','sample_submission.csv'):
                DATA_FILES.setdefault(f, os.path.join(root, f))
    assert len(DATA_FILES) == 3, DATA_FILES
    train = pd.read_csv(DATA_FILES['train.csv'])
    test = pd.read_csv(DATA_FILES['test.csv'])
    sample_sub = pd.read_csv(DATA_FILES['sample_submission.csv'])

assert list(train.columns) == ['id','title','content','label']
print("train:", train.shape, " test:", test.shape)

def normalize_for_hash(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    return re.sub(r'\s+',' ',s).strip()

for df in (train, test):
    df['content_hash'] = df['content'].apply(lambda s: hashlib.md5(normalize_for_hash(s).encode()).hexdigest())
    df['title_hash']   = df['title'].apply(lambda s: hashlib.md5(normalize_for_hash(s).encode()).hexdigest())

with Timer("exact_pair_overlap"):
    exact_pair_map = (train.drop_duplicates(subset=['title_hash','content_hash'])
                           .set_index(['title_hash','content_hash'])['label'].to_dict())
    test_override_keys = list(zip(test['title_hash'], test['content_hash']))
    n_overridden_test = sum(1 for k in test_override_keys if k in exact_pair_map)
    # verified in offline analysis: 0 exact pairs ever carry conflicting labels -> override is sound
    print(f"test rows with an exact (title,content) match in train: {n_overridden_test}/{len(test)} "
          f"({n_overridden_test/len(test)*100:.1f}%)")

## 4. Leakage-Safe Split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
with Timer("split"):
    gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['val_size'], random_state=SEED)
    train_idx, val_idx = next(gss.split(train, train['label'], groups=train['content_hash']))
    assert not (set(train['content_hash'].iloc[train_idx]) & set(train['content_hash'].iloc[val_idx]))
    print(f"train rows: {len(train_idx)}   val rows: {len(val_idx)}   "
          f"(val class-0 share: {(train['label'].iloc[val_idx]==0).mean():.3f})")

## 5. Text Cleaning

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
HTML_RE = re.compile(r'<[^>]+>')
WS_RE = re.compile(r'\s+')

def clean_text(s):
    s = unicodedata.normalize('NFKC', str(s))
    s = HTML_RE.sub(' ', s); s = URL_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

with Timer("text_cleaning"):
    for df in (train, test):
        df['title_clean'] = df['title'].apply(clean_text)
        df['content_clean'] = df['content'].apply(clean_text)

## 6. Entity Mining (purity filter + distributional type classes)

Built from **train content only**. The purity filter is what makes this work: a genuine
proper noun almost never appears lowercase, while common Indonesian words that happen to
be capitalized mid-headline do.

In [ ]:
with Timer("entity_mining"):
    upper_c, lower_c = Counter(), Counter()
    for txt in train['content_clean']:
        for t in txt.split():
            c = t.strip(_string.punctuation)
            if not c.isalpha() or len(c) < 3:
                continue
            if c[:1].isupper(): upper_c[c] += 1
            else: lower_c[c.capitalize()] += 1

    purity = {w: upper_c[w]/(upper_c[w]+lower_c.get(w,0))
              for w in upper_c if upper_c[w] >= CONFIG['entity_min_count']}

    title_df = Counter()
    for t in train['title_clean']:
        for w in set(t.split()): title_df[w] += 1
    N = len(train)

    ENT = {w for w,p in purity.items()
           if p >= CONFIG['entity_min_purity'] and title_df.get(w,0)/N < CONFIG['entity_max_title_df']}
    print(f"entity pool: {len(ENT)}")
    for w in ['DKI','Jabar','Jokowi','Bogor','Tidak','Jenazah','Corona','Presiden']:
        if w in purity:
            print(f"   {w:10s} purity={purity[w]:.2f}  in_pool={w in ENT}")

    # distributional type classes: entities sharing a preceding headline word are swap-compatible
    prev_ctx = defaultdict(Counter)
    for t in train['title_clean']:
        toks = t.split()
        for i,w in enumerate(toks):
            if w in ENT and i > 0: prev_ctx[w][toks[i-1]] += 1
    ctx_to_ents = defaultdict(set)
    for e,ctxs in prev_ctx.items():
        for c,n in ctxs.items():
            if n >= 2: ctx_to_ents[c].add(e)
    print(f"context (type) groups: {len(ctx_to_ents)}")
    for c in ['Gubernur','Pemkot','Kapolda']:
        if c in ctx_to_ents:
            print(f"   after '{c}': {sorted(ctx_to_ents[c])[:10]}")

    ENT_LIST = sorted(ENT)
    _freq = np.array([upper_c[w] for w in ENT_LIST], dtype=float)
    ENT_P = _freq/_freq.sum()

## 7. Synthetic Negative Generation

Applied to **train_idx positives only**. Constraints that keep the generated label
trustworthy:
- the swapped-**out** token must be a pool entity that *is* present in the body (so a
  grounded claim is being falsified)
- the swapped-**in** entity must *not* appear anywhere in the body (otherwise the new
  headline could still be supported, which would be a mislabeled example)

In [ ]:
def _type_pool(toks, i):
    pool = set()
    if i > 0 and toks[i-1] in ctx_to_ents: pool |= ctx_to_ents[toks[i-1]]
    if i+1 < len(toks) and toks[i+1] in ctx_to_ents: pool |= ctx_to_ents[toks[i+1]]
    return pool

def generate_synthetic_negative(title, content, rng):
    """Mimic the organizers' negative construction.

    Measured on real data, only ~25-38% of real negatives contain a title entity that is
    absent from the body -- meaning the majority swap in an entity that IS present in the
    body (a wrong-attribution/role error, e.g. 'Pemkot Surabaya ... Pemprov Jatim' with
    the two reversed). Generating only the out-of-body kind would make the
    'entity missing from body' conflict flag a perfect giveaway on synthetic rows
    (measured: fires on 100% of them) and teach a shortcut that does not transfer.
    So we deliberately produce a mix.
    """
    ctoks = set(w.strip(_string.punctuation) for w in content.split())
    toks = title.split()
    grounded = [i for i,w in enumerate(toks) if w in ENT and w in ctoks]
    if not grounded:
        return None

    # --- Type C: role reversal between two grounded title entities (all entities stay
    # in-body, so the missing-entity flag does not fire) ---
    if len(grounded) >= 2 and rng.random() < 0.50:
        i, j = rng.choice(grounded, size=2, replace=False)
        if toks[i] != toks[j]:
            nt = toks.copy(); nt[i], nt[j] = nt[j], nt[i]
            return ' '.join(nt)

    rng.shuffle(grounded)
    # --- Type B (~65%): swap in a type-compatible entity that IS present in the body ---
    # --- Type A (~35%): swap in one that is NOT present (the easy, flag-tripping kind) ---
    want_in_body = rng.random() < 0.65
    body_ents = sorted({w for w in ctoks if w in ENT})
    for i in grounded:
        tp = _type_pool(toks, i)
        in_body  = sorted({w for w in tp if w in ctoks and w != toks[i]})
        off_body = sorted({w for w in tp if w not in ctoks and w != toks[i]})
        # when no type-compatible in-body entity exists, prefer ANY other entity that is
        # present in the body over falling back to an out-of-body one -- co-occurring
        # entities are still plausible confusions, and this keeps the synthetic
        # 'entity missing from body' rate close to what real negatives show
        if want_in_body and not in_body:
            in_body = [w for w in body_ents if w != toks[i] and w not in toks]
        primary, fallback = (in_body, off_body) if want_in_body else (off_body, in_body)
        if primary:
            return ' '.join(toks[:i] + [rng.choice(primary)] + toks[i+1:])
        if fallback:
            return ' '.join(toks[:i] + [rng.choice(fallback)] + toks[i+1:])
        # no type-compatible candidate: fall back to a frequency-sampled entity
        for _ in range(30):
            w = ENT_LIST[rng.choice(len(ENT_LIST), p=ENT_P)]
            if w != toks[i] and (w in ctoks) == want_in_body:
                return ' '.join(toks[:i] + [w] + toks[i+1:])
    return None

def make_synthetic_set(source_df, seed, label_for_log):
    """Generate synthetic negatives from the positives of source_df."""
    rng = np.random.default_rng(seed)
    n_pos = int((source_df['label']==1).sum()); n_neg = int((source_df['label']==0).sum())
    target = CONFIG['target_neg_share']
    # solve (n_neg + k) / (n_pos + n_neg + k) = target
    n_needed = max(0, int(round((target*(n_pos+n_neg) - n_neg)/(1-target))))
    pos_pool = source_df[source_df['label']==1].sample(frac=1.0, random_state=seed)
    rows, attempts = [], 0
    for _, r in pos_pool.iterrows():
        if len(rows) >= n_needed: break
        attempts += 1
        fake = generate_synthetic_negative(r['title_clean'], r['content_clean'], rng)
        if fake is None: continue
        rows.append({'title_clean': fake, 'content_clean': r['content_clean'],
                     'content_hash': r['content_hash'],
                     'title_hash': hashlib.md5(normalize_for_hash(fake).encode()).hexdigest(),
                     'label': 0})
    out = pd.DataFrame(rows)
    print(f"[{label_for_log}] {n_pos} pos / {n_neg} neg -> wanted {n_needed}, generated {len(out)} "
          f"from {attempts} attempts ({len(out)/max(attempts,1)*100:.0f}% success)")
    return out

with Timer("synthetic_negative_generation"):
    # Both sets are generated up front so that chunk/title encoding in Section 8 happens
    # exactly once for everything (re-encoding ~250k chunks later would cost 5-10 GPU min).
    synth = make_synthetic_set(train.iloc[train_idx], SEED, 'validation-run')
    synth_all = make_synthetic_set(train, SEED+1, 'final-retrain')
    if len(synth):
        for k in range(min(3, len(synth))):
            print(f"   FAKE: {synth.iloc[k]['title_clean']}")

## 8. Hybrid Overlapping Retrieval + Conflict Text

In [ ]:
NUM_RE = re.compile(r'\d+[.,]?\d*\s*%?')

def body_entities(s):
    toks = s.split(); ents = set()
    for i,t in enumerate(toks):
        c = t.strip(_string.punctuation)
        if i > 0 and c[:1].isupper() and c.isalpha() and len(c) > 2:
            ents.add(c.lower())
    return ents

def build_conflict_text(title, content):
    ctoks_raw = set(w.strip(_string.punctuation) for w in content.split())
    c_ents = body_entities(content)
    title_l = title.lower()
    parts = []
    # THE discriminative signal (38% of negatives vs 9.8% of positives): title entities
    # that never appear in the body -- i.e. the fabricated one
    title_ents_missing = [w for w in title.split() if w in ENT and w not in ctoks_raw][:5]
    if title_ents_missing:
        parts.append('ENTITAS JUDUL TAK ADA DI ISI: ' + ', '.join(title_ents_missing))
    tnum = set(NUM_RE.findall(title)); cnum = set(NUM_RE.findall(content))
    num_conflict = list(tnum - cnum)[:5]
    if num_conflict:
        parts.append('ANGKA JUDUL TAK ADA DI ISI: ' + ', '.join(num_conflict))
    missing_in_title = [e for e in c_ents if e not in title_l][:4]
    if missing_in_title:
        parts.append('ENTITAS ISI: ' + ', '.join(missing_in_title))
    return ' | '.join(parts)

In [ ]:
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer

def chunk_overlapping(s, chunk_words=CONFIG['chunk_words'], stride=CONFIG['chunk_stride']):
    words = s.split()
    if not words: return []
    if len(words) <= chunk_words: return [' '.join(words)]
    chunks = []
    for i in range(0, len(words), stride):
        ch = words[i:i+chunk_words]
        if not ch: break
        chunks.append(' '.join(ch))
        if i + chunk_words >= len(words): break
    return chunks

with Timer("retriever_load"):
    retr_tok = AutoTokenizer.from_pretrained(CONFIG['retriever_model'])
    retr_model = AutoModel.from_pretrained(CONFIG['retriever_model']).to(DEVICE).eval()

def average_pool(last_hidden, attention_mask):
    last_hidden = last_hidden.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1)/attention_mask.sum(dim=1)[..., None]

@torch.no_grad()
def encode_texts(texts, prefix, batch_size=None, max_length=None):
    batch_size = batch_size or CONFIG['retriever_batch_size']
    max_length = max_length or CONFIG['retriever_max_len']
    outs = []; use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        b = [prefix + t for t in texts[i:i+batch_size]]
        enc = retr_tok(b, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            o = retr_model(**enc)
        e = average_pool(o.last_hidden_state, enc['attention_mask'])
        e = torch.nn.functional.normalize(e, p=2, dim=1)
        outs.append(e.float().cpu().numpy())
    return np.concatenate(outs, axis=0) if outs else np.zeros((0, retr_model.config.hidden_size))

In [ ]:
with Timer("retrieval_encode"):
    _c = ['content_hash','title_hash','content_clean','title_clean']
    all_frames = [train[_c], test[_c]]
    if len(synth): all_frames.append(synth[_c])
    if len(synth_all): all_frames.append(synth_all[_c])
    combined = pd.concat(all_frames, ignore_index=True)

    uniq_content = combined.drop_duplicates('content_hash')[['content_hash','content_clean']].reset_index(drop=True)
    flat_chunks, chunk_ranges = [], {}
    for h, ctext in zip(uniq_content['content_hash'], uniq_content['content_clean']):
        chs = chunk_overlapping(ctext)
        start = len(flat_chunks)
        flat_chunks.extend(chs if chs else [''])
        chunk_ranges[h] = (start, len(flat_chunks))
    print(f"unique articles: {len(uniq_content)}   chunks: {len(flat_chunks)}")

    chunk_embs_flat = encode_texts(flat_chunks, prefix='passage: ')
    uniq_title = combined.drop_duplicates('title_hash')[['title_hash','title_clean']].reset_index(drop=True)
    title_embs = encode_texts(uniq_title['title_clean'].tolist(), prefix='query: ', max_length=32)
    title_emb_map = dict(zip(uniq_title['title_hash'], title_embs))
    print(f"titles encoded (incl. synthetic): {len(uniq_title)}")

with Timer("lexical_tfidf"):
    lex_vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True,
                               max_features=CONFIG['lexical_max_features'])
    lex_vec.fit(pd.concat([uniq_title['title_clean'], pd.Series(flat_chunks)]))
    chunk_tfidf_flat = lex_vec.transform(flat_chunks)
    title_tfidf_map = dict(zip(uniq_title['title_hash'], lex_vec.transform(uniq_title['title_clean'])))

In [ ]:
def retrieve_hybrid(df):
    out = []
    for h, th in zip(df['content_hash'], df['title_hash']):
        s, e = chunk_ranges[h]
        ctexts = flat_chunks[s:e]; n = len(ctexts)
        sem = chunk_embs_flat[s:e] @ title_emb_map[th]
        lex = np.asarray((chunk_tfidf_flat[s:e] @ title_tfidf_map[th].T).todense()).ravel()
        sel = set(np.argsort(-sem)[:min(CONFIG['top_k_semantic'], n)].tolist()) | \
              set(np.argsort(-lex)[:min(CONFIG['top_k_lexical'], n)].tolist())
        if CONFIG['include_first_chunk']: sel.add(0)
        out.append(' '.join(ctexts[i] for i in sorted(sel)))
    return out

with Timer("retrieval_apply"):
    for df in ([train, test] + ([synth] if len(synth) else []) + ([synth_all] if len(synth_all) else [])):
        df['retrieved_text'] = retrieve_hybrid(df)
        df['conflict_text'] = [build_conflict_text(t, c) for t, c in zip(df['title_clean'], df['content_clean'])]
        df['model_input_body'] = df['retrieved_text'] + df['conflict_text'].apply(lambda s: (' [SEP] '+s) if s else '')

del chunk_embs_flat, chunk_tfidf_flat

# sanity: the discriminative conflict signal should fire far more often on real negatives
_has = train['conflict_text'].str.contains('ENTITAS JUDUL TAK ADA DI ISI')
print("rate of 'title entity missing from body' by label (real train rows):")
print(pd.DataFrame({'flag': _has, 'label': train['label']}).groupby('label')['flag'].mean())
if len(synth):
    print("same flag on synthetic negatives:",
          round(synth['conflict_text'].str.contains('ENTITAS JUDUL TAK ADA DI ISI').mean(), 3))

## 9. Build Augmented Training Set

In [ ]:
with Timer("assemble_training_set"):
    cols = ['title_clean','model_input_body','label']
    real_tr = train.iloc[train_idx][cols]
    aug_tr = pd.concat([real_tr] + ([synth[cols]] if len(synth) else []), ignore_index=True)
    aug_tr = aug_tr.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    val_titles = train['title_clean'].iloc[val_idx]
    val_bodies = train['model_input_body'].iloc[val_idx]
    val_labels = train['label'].iloc[val_idx].values

    print(f"augmented training set: {len(aug_tr)} rows "
          f"(class-0 share {1-aug_tr['label'].mean():.3f}); validation kept at "
          f"{len(val_labels)} rows (class-0 share {(val_labels==0).mean():.3f}, i.e. the real distribution)")

## 10. Training (class-weighted, FP16, early stopping on Macro F1)

In [ ]:
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments,
                           EarlyStoppingCallback)
from sklearn.metrics import f1_score, confusion_matrix

class PairDataset(Dataset):
    def __init__(self, titles, bodies, labels, tokenizer, max_len):
        self.t = list(titles); self.b = list(bodies)
        self.y = None if labels is None else list(labels)
        self.tok = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        enc = self.tok(self.t[i], self.b[i], truncation=True, max_length=self.max_len,
                        padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.y is not None: item['labels'] = torch.tensor(self.y[i], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=self.class_weights.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def make_class_weights(labels):
    n = len(labels); n0 = (labels==0).sum(); n1 = (labels==1).sum()
    return torch.tensor([n/(2*n0), n/(2*n1)], dtype=torch.float)

def build_trainer(train_titles, train_bodies, train_labels, val_titles, val_bodies, val_labels,
                   epochs, out_dir, early_stopping=True):
    tok = AutoTokenizer.from_pretrained(CONFIG['classifier_model'])
    model = AutoModelForSequenceClassification.from_pretrained(CONFIG['classifier_model'], num_labels=2)
    tr_ds = PairDataset(train_titles, train_bodies, train_labels, tok, CONFIG['max_len'])
    cw = make_class_weights(np.array(train_labels))
    print(f"class weights: {cw.tolist()}")
    common = dict(output_dir=out_dir, num_train_epochs=epochs,
                  per_device_train_batch_size=CONFIG['batch_size'],
                  learning_rate=CONFIG['lr'], weight_decay=0.01,
                  fp16=torch.cuda.is_available(), dataloader_num_workers=0, seed=SEED,
                  logging_steps=100, report_to=[], disable_tqdm=False)
    if early_stopping:
        args = TrainingArguments(per_device_eval_batch_size=CONFIG['batch_size']*2,
                                 eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
                                 load_best_model_at_end=True, metric_for_best_model='macro_f1',
                                 greater_is_better=True, **common)
        return WeightedTrainer(model=model, args=args, train_dataset=tr_ds,
                               eval_dataset=PairDataset(val_titles, val_bodies, val_labels, tok, CONFIG['max_len']),
                               compute_metrics=compute_metrics, class_weights=cw,
                               callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])]), tok
    args = TrainingArguments(eval_strategy='no', save_strategy='no', **common)
    return WeightedTrainer(model=model, args=args, train_dataset=tr_ds, class_weights=cw), tok

In [ ]:
with Timer("training"):
    t0 = time.time()
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    trainer, tok = build_trainer(aug_tr['title_clean'], aug_tr['model_input_body'], aug_tr['label'].values,
                                  val_titles, val_bodies, val_labels,
                                  epochs=CONFIG['epochs'], out_dir='/kaggle/working/ckpt')
    trainer.train()
    train_time = time.time() - t0
    peak_mem = gpu_mem_mb()

val_out = trainer.predict(trainer.eval_dataset)
val_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()[:,1]
print(f"training time: {train_time:.1f}s   peak GPU: {peak_mem:.0f} MB")
print(f"Macro F1 @0.5: {f1_score(val_labels, (val_probs>=0.5).astype(int), average='macro'):.4f}")
print("confusion @0.5:\n", confusion_matrix(val_labels, (val_probs>=0.5).astype(int)))
rm_checkpoint('/kaggle/working/ckpt')

## 11. Threshold Tuning

In [ ]:
with Timer("threshold_tuning"):
    rows = []
    for thr in CONFIG['threshold_grid']:
        p = (val_probs >= thr).astype(int)
        rows.append({'threshold': thr,
                     'f1_class0': f1_score(val_labels, p, pos_label=0, zero_division=0),
                     'f1_class1': f1_score(val_labels, p, pos_label=1, zero_division=0),
                     'macro_f1': f1_score(val_labels, p, average='macro'),
                     'pos_rate': p.mean()})
    thr_df = pd.DataFrame(rows)
    BEST_THRESHOLD = float(thr_df.loc[thr_df['macro_f1'].idxmax(),'threshold'])
    BEST_VAL_MACRO_F1 = float(thr_df['macro_f1'].max())
print(thr_df.to_string(index=False))
print(f"\nBest threshold {BEST_THRESHOLD} -> Macro F1 {BEST_VAL_MACRO_F1:.4f}")
print("confusion at best threshold:\n",
      confusion_matrix(val_labels, (val_probs>=BEST_THRESHOLD).astype(int)))

## 12. Final Training on Full Data

Re-runs the whole recipe on all of `train` plus synthetic negatives generated from all of
`train` (same generator, same constraints), for the epoch count the validated run reached.

In [ ]:
with Timer("final_training"):
    t0 = time.time()
    # Retrain for the epoch that actually scored best, NOT however many epochs ran before
    # early stopping fired -- with patience=1 the run always goes one epoch past the optimum.
    _hist = [h for h in trainer.state.log_history if 'eval_macro_f1' in h]
    if _hist:
        _best = max(_hist, key=lambda h: h['eval_macro_f1'])
        final_epochs = max(1, int(round(_best.get('epoch', CONFIG['epochs']))))
        print(f"best eval epoch was {final_epochs} (macro_f1={_best['eval_macro_f1']:.4f}); "
              f"total epochs run: {trainer.state.epoch}")
    else:
        final_epochs = CONFIG['epochs']
    cols = ['title_clean','model_input_body','label']
    full_tr = pd.concat([train[cols]] + ([synth_all[cols]] if len(synth_all) else []), ignore_index=True)
    full_tr = full_tr.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(f"final training set: {len(full_tr)} rows (class-0 share {1-full_tr['label'].mean():.3f}), "
          f"epochs={final_epochs}")
    final_trainer, final_tok = build_trainer(full_tr['title_clean'], full_tr['model_input_body'],
                                              full_tr['label'].values, None, None, None,
                                              epochs=final_epochs, out_dir='/kaggle/working/final_ckpt',
                                              early_stopping=False)
    final_trainer.train()
    final_train_time = time.time() - t0
print(f"final model trained in {final_train_time:.1f}s")

## 13. Test Inference & Submission

In [ ]:
with Timer("test_inference"):
    test_ds = PairDataset(test['title_clean'], test['model_input_body'], None, final_tok, CONFIG['max_len'])
    test_probs = torch.softmax(torch.tensor(final_trainer.predict(test_ds).predictions), dim=-1).numpy()[:,1]
    test_pred = (test_probs >= BEST_THRESHOLD).astype(int)
print(f"model positive-rate on test: {test_pred.mean():.3f}")

with Timer("submission"):
    final_pred = test_pred.copy()
    for i, key in enumerate(test_override_keys):
        if key in exact_pair_map:
            final_pred[i] = exact_pair_map[key]

    submission = pd.DataFrame({'id': test['id'], 'label': final_pred.astype(int)})
    submission = submission.set_index('id').loc[test['id']].reset_index()
    assert submission.shape[0] == len(test)
    assert set(submission['id']) == set(sample_sub['id'])
    assert set(submission['label'].unique()).issubset({0,1})
    assert list(submission.columns) == ['id','label']
    submission.to_csv('/kaggle/working/submission.csv', index=False)

print(submission['label'].value_counts(normalize=True))
print(f"exact-match override applied to {n_overridden_test} rows; final positive-rate {final_pred.mean():.3f}")

In [ ]:
TOTAL = time.time() - RUN_T0
print("="*55)
print("v6 — synthetic negative augmentation")
print(f"Validation Macro F1: {BEST_VAL_MACRO_F1:.4f} @ threshold {BEST_THRESHOLD}")
print(f"Synthetic negatives: {len(synth)} (val run) / {len(synth_all)} (final)")
print(f"Peak GPU: {peak_mem:.0f} MB   Total runtime: {TOTAL/60:.1f} min")
print("Submission: /kaggle/working/submission.csv")
print("="*55)
for k,v in sorted(TIMINGS.items(), key=lambda kv:-kv[1]):
    print(f"  {k}: {v:.1f}s")